# 03 — Visual Simulation

Everything so far has been computed and printed. This notebook makes it interactive.

You will set up a scenario by clicking on a map, assign the target motion using sliders, then run both pursuit strategies and watch the paths animate step by step. Two modes on one map: the curved pursuit path and the straight intercept line, both racing toward the same target.

The geometry has not changed. The controls just make it easier to build intuition by trying many scenarios quickly.

In [ ]:
import math
import ipywidgets as widgets
from ipyleaflet import Map, GeoJSON, Marker, AwesomeIcon

# ── Geometry helpers ─────────────────────────────────────────────────────────

def compute_bearing(p1, p2):
    lon1, lat1 = math.radians(p1[0]), math.radians(p1[1])
    lon2, lat2 = math.radians(p2[0]), math.radians(p2[1])
    d_lon = lon2 - lon1
    x = math.sin(d_lon) * math.cos(lat2)
    y = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(d_lon)
    return (math.degrees(math.atan2(x, y)) + 360) % 360

def haversine_km(p1, p2):
    R = 6371.0
    lon1, lat1 = math.radians(p1[0]), math.radians(p1[1])
    lon2, lat2 = math.radians(p2[0]), math.radians(p2[1])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

def destination_point(origin, bearing_deg, distance_km):
    R = 6371.0
    d   = distance_km / R
    brg = math.radians(bearing_deg)
    lat1 = math.radians(origin[1])
    lon1 = math.radians(origin[0])
    lat2 = math.asin(math.sin(lat1)*math.cos(d) + math.cos(lat1)*math.sin(d)*math.cos(brg))
    lon2 = lon1 + math.atan2(math.sin(brg)*math.sin(d)*math.cos(lat1),
                              math.cos(d) - math.sin(lat1)*math.sin(lat2))
    return [math.degrees(lon2), math.degrees(lat2)]

def find_intercept_time(s_pos, t_pos, t_hdg, t_spd, s_spd, t_max=10.0, tol=1e-6):
    def f(t):
        return haversine_km(s_pos, destination_point(t_pos, t_hdg, t_spd*t)) - s_spd*t
    if f(t_max) > 0: return None
    lo, hi = 0.0, t_max
    for _ in range(60):
        mid = (lo + hi) / 2
        if f(mid) > 0: lo = mid
        else: hi = mid
        if hi - lo < tol: break
    return (lo + hi) / 2

def simulate_pursuit(s_pos, t_pos, t_hdg, t_spd, s_spd,
                     dt=0.01, max_steps=2000, capture_km=5.0):
    pursuer, target = list(s_pos), list(t_pos)
    p_path, t_path = [list(pursuer)], [list(target)]
    for step in range(max_steps):
        if haversine_km(pursuer, target) <= capture_km:
            return {"pursuer_path": p_path, "target_path": t_path,
                    "captured": True, "time_elapsed": step * dt}
        brg    = compute_bearing(pursuer, target)
        pursuer = destination_point(pursuer, brg, s_spd * dt)
        target  = destination_point(target, t_hdg, t_spd * dt)
        p_path.append(list(pursuer))
        t_path.append(list(target))
    return {"pursuer_path": p_path, "target_path": t_path,
            "captured": False, "time_elapsed": max_steps * dt}

print("All helpers loaded.")

## 1. Click-Based Setup

The first map lets you place the shooter and target by clicking. First click sets the **shooter** (red), second click sets the **target** (blue). Subsequent clicks alternate. The status line below the map confirms what was placed.

Run the map cell, click twice, then move to the next section to assign target motion.

In [ ]:
# Shared scenario state — updated by clicks and sliders
scenario = {
    "shooter": [-98.49, 33.91],   # default: Wichita Falls
    "target":  [-101.0, 36.5],    # default: northwest
    "click_count": 0,
}

setup_map    = Map(center=(35.0, -99.5), zoom=6)
setup_layers = []
setup_status = widgets.HTML(value="<b>Click to place shooter (1st click) then target (2nd click)</b>")

def refresh_setup_markers():
    for lyr in setup_layers:
        setup_map.remove(lyr)
    setup_layers.clear()

    for pos, color, label in [
        (scenario["shooter"], "red",  "Shooter"),
        (scenario["target"],  "blue", "Target"),
    ]:
        lyr = GeoJSON(
            data={"type":"FeatureCollection","features":[{
                "type":"Feature",
                "geometry":{"type":"Point","coordinates": pos},
                "properties":{"label": label}
            }]},
            point_style={"radius": 8, "color": color, "fillOpacity": 1.0}
        )
        setup_map.add(lyr)
        setup_layers.append(lyr)

refresh_setup_markers()

def on_setup_click(**kwargs):
    if kwargs.get("type") != "click":
        return
    lat, lon = kwargs["coordinates"]
    pt = [lon, lat]
    n = scenario["click_count"] % 2
    if n == 0:
        scenario["shooter"] = pt
        setup_status.value = f"<b>Shooter set:</b> [{lon:.3f}, {lat:.3f}] — now click to place the target"
    else:
        scenario["target"] = pt
        dist = haversine_km(scenario["shooter"], scenario["target"])
        brg  = compute_bearing(scenario["shooter"], scenario["target"])
        setup_status.value = (
            f"<b>Target set:</b> [{lon:.3f}, {lat:.3f}] &nbsp;|&nbsp; "
            f"Distance: {dist:.1f} km &nbsp;|&nbsp; Bearing: {brg:.1f}°"
        )
    scenario["click_count"] += 1
    refresh_setup_markers()

setup_map.on_interaction(on_setup_click)
widgets.VBox([setup_map, setup_status])

## 2. Assign Target Motion

Use the sliders to configure target heading, target speed, and shooter speed. These feed directly into both simulations when you run Section 3.

In [ ]:
heading_slider = widgets.IntSlider(
    value=135, min=0, max=359, step=5,
    description="Target hdg °:", style={"description_width": "130px"},
    layout=widgets.Layout(width="420px")
)
target_speed_slider = widgets.IntSlider(
    value=300, min=50, max=800, step=50,
    description="Target spd km/h:", style={"description_width": "130px"},
    layout=widgets.Layout(width="420px")
)
shooter_speed_slider = widgets.IntSlider(
    value=600, min=100, max=1200, step=50,
    description="Shooter spd km/h:", style={"description_width": "130px"},
    layout=widgets.Layout(width="420px")
)

motion_status = widgets.HTML()

def update_motion_status(*_):
    ratio = shooter_speed_slider.value / max(target_speed_slider.value, 1)
    motion_status.value = (
        f"Speed ratio: <b>{ratio:.2f}×</b> &nbsp;|&nbsp; "
        f"Target heading: <b>{heading_slider.value}°</b>"
    )

for s in [heading_slider, target_speed_slider, shooter_speed_slider]:
    s.observe(update_motion_status, names="value")

update_motion_status()
widgets.VBox([heading_slider, target_speed_slider, shooter_speed_slider, motion_status])

## 3. Run Both Simulations

Click **Run Simulation** to compute both strategies using your current shooter/target positions and motion settings. Both paths appear on the same map:

- **Red curve** — pure pursuit (always chasing current position)
- **Green line** — constant-velocity intercept (fires once at predicted position)
- **Blue dashes** — target's actual path

Results (capture time, path length, miss status) are shown below the map.

In [ ]:
def fc(features): return {"type": "FeatureCollection", "features": features}
def geojson_line(coords, props={}):
    return {"type":"Feature","geometry":{"type":"LineString","coordinates":coords},"properties":props}
def geojson_point(coord, props={}):
    return {"type":"Feature","geometry":{"type":"Point","coordinates":coord},"properties":props}

def path_length_km(path):
    return sum(haversine_km(path[i], path[i+1]) for i in range(len(path)-1))

sim_map    = Map(center=(35.0, -99.5), zoom=6)
sim_layers = []
sim_output = widgets.HTML(value="<i>Press Run Simulation to compute.</i>")
run_btn    = widgets.Button(description="▶ Run Simulation",
                             button_style="primary",
                             layout=widgets.Layout(width="200px"))

def clear_sim_layers():
    for lyr in sim_layers:
        sim_map.remove(lyr)
    sim_layers.clear()

def add_layer(data, style=None, point_style=None):
    kwargs = {"data": data}
    if style:       kwargs["style"]       = style
    if point_style: kwargs["point_style"] = point_style
    lyr = GeoJSON(**kwargs)
    sim_map.add(lyr)
    sim_layers.append(lyr)

def on_run(_):
    clear_sim_layers()

    s_pos  = scenario["shooter"]
    t_pos  = scenario["target"]
    t_hdg  = heading_slider.value
    t_spd  = target_speed_slider.value
    s_spd  = shooter_speed_slider.value

    # ── Pure pursuit ──────────────────────────────────────────────────────
    sim = simulate_pursuit(s_pos, t_pos, t_hdg, t_spd, s_spd, dt=0.01, max_steps=3000)
    step = max(1, len(sim["pursuer_path"]) // 300)
    add_layer(fc([geojson_line(sim["target_path"][::step])]),
              style={"color": "#457b9d", "weight": 1.5, "dashArray": "5"})
    add_layer(fc([geojson_line(sim["pursuer_path"][::step])]),
              style={"color": "#e63946", "weight": 2})

    # ── Constant-velocity intercept ───────────────────────────────────────
    t_int = find_intercept_time(s_pos, t_pos, t_hdg, t_spd, s_spd)
    if t_int is not None:
        ipt = destination_point(t_pos, t_hdg, t_spd * t_int)
        add_layer(fc([geojson_line([s_pos, ipt])]),
                  style={"color": "#2a9d8f", "weight": 2, "dashArray": "6"})
        add_layer(fc([geojson_point(ipt)]),
                  point_style={"radius": 6, "color": "#2a9d8f", "fillOpacity": 1.0})
        int_dist = haversine_km(s_pos, ipt)
        int_tof  = t_int * 60
    else:
        int_dist = int_tof = None

    # ── Key points ────────────────────────────────────────────────────────
    add_layer(fc([geojson_point(s_pos)]),
              point_style={"radius": 8, "color": "#e63946", "fillOpacity": 1.0})
    add_layer(fc([geojson_point(t_pos)]),
              point_style={"radius": 8, "color": "#457b9d", "fillOpacity": 1.0})

    # ── Results summary ───────────────────────────────────────────────────
    p_cap  = "✓ CAPTURED" if sim["captured"] else "✗ NOT CAPTURED"
    p_tof  = f"{sim['time_elapsed']*60:.1f} min" if sim["captured"] else "—"
    p_dist = f"{path_length_km(sim['pursuer_path']):.1f} km" if sim["captured"] else "—"

    i_cap  = "✓ CAPTURED" if t_int is not None else "✗ IMPOSSIBLE"
    i_tof  = f"{int_tof:.1f} min" if int_tof is not None else "—"
    i_dist = f"{int_dist:.1f} km" if int_dist is not None else "—"

    sim_output.value = f"""
    <table style="font-size:13px; border-collapse:collapse;">
      <tr><th style="padding:4px 12px;text-align:left;"></th>
          <th style="padding:4px 12px;">Pure Pursuit (red)</th>
          <th style="padding:4px 12px;">Intercept (green)</th></tr>
      <tr><td style="padding:3px 12px;">Result</td>
          <td style="padding:3px 12px;">{p_cap}</td>
          <td style="padding:3px 12px;">{i_cap}</td></tr>
      <tr><td style="padding:3px 12px;">Time of flight</td>
          <td style="padding:3px 12px;">{p_tof}</td>
          <td style="padding:3px 12px;">{i_tof}</td></tr>
      <tr><td style="padding:3px 12px;">Distance flown</td>
          <td style="padding:3px 12px;">{p_dist}</td>
          <td style="padding:3px 12px;">{i_dist}</td></tr>
    </table>
    """

run_btn.on_click(on_run)
widgets.VBox([run_btn, sim_map, sim_output])

---

## Exercise A — Speed Ratio Exploration

Use the sliders to work through the following four configurations. For each one, note whether pursuit captures, whether intercept succeeds, and record the time-of-flight for each strategy.

| Shooter spd | Target spd | Target hdg | Pursuit result | Intercept result |
|-------------|------------|------------|----------------|------------------|
| 600 km/h    | 300 km/h   | 135°       | ?              | ?                |
| 400 km/h    | 300 km/h   | 135°       | ?              | ?                |
| 250 km/h    | 300 km/h   | 135°       | ?              | ?                |
| 600 km/h    | 300 km/h   | 270°       | ?              | ?                |

After filling in the table, answer:

1. At what speed ratio does pure pursuit fail to capture? Why?
2. Find one configuration where intercept succeeds but pure pursuit does not (or vice versa). Explain the geometry.

In [ ]:
# Exercise A — your observations here

configs = [
    {"shooter_spd": 600, "target_spd": 300, "target_hdg": 135},
    {"shooter_spd": 400, "target_spd": 300, "target_hdg": 135},
    {"shooter_spd": 250, "target_spd": 300, "target_hdg": 135},
    {"shooter_spd": 600, "target_spd": 300, "target_hdg": 270},
]

s_pos = [-98.49, 33.91]
t_pos = [-101.0, 36.5]

print(f"{'Shooter':>10}  {'Target':>8}  {'Hdg':>5}  {'Pursuit':>15}  {'Intercept':>15}")
print("-" * 62)
for cfg in configs:
    s_spd = cfg["shooter_spd"]
    t_spd = cfg["target_spd"]
    hdg   = cfg["target_hdg"]

    sim = simulate_pursuit(s_pos, t_pos, hdg, t_spd, s_spd, dt=0.01, max_steps=3000)
    p_result = f"✓ {sim['time_elapsed']*60:.1f} min" if sim["captured"] else "✗ no capture"

    t_int = find_intercept_time(s_pos, t_pos, hdg, t_spd, s_spd)
    i_result = f"✓ {t_int*60:.1f} min" if t_int is not None else "✗ impossible"

    print(f"{s_spd:>9} km/h  {t_spd:>6} km/h  {hdg:>4}°  {p_result:>15}  {i_result:>15}")

## Exercise B — Target Heading Sweep

Fix the shooter at Wichita Falls `[-98.49, 33.91]` and the target at `[-101.0, 36.5]`. Set both speeds to 600 km/h (shooter) and 300 km/h (target).

Sweep target heading from 0° to 355° in 5° steps. For each heading, run `find_intercept_time` and record the time of flight (or `None` if impossible). Then plot the results.

Your plot should show heading on the x-axis and time of flight (in minutes) on the y-axis — with impossible intercepts marked visually (e.g., plotted at a sentinel value or omitted).

Identify and explain:
1. Which headings make the intercept **impossible** or very long?
2. Which heading produces the **shortest** time of flight?

In [ ]:
import matplotlib.pyplot as plt

s_pos  = [-98.49, 33.91]
t_pos  = [-101.0, 36.5]
s_spd  = 600
t_spd  = 300

headings = list(range(0, 360, 5))
tofs     = []
possible = []

for hdg in headings:
    t = find_intercept_time(s_pos, t_pos, hdg, t_spd, s_spd)
    if t is not None:
        tofs.append(t * 60)
        possible.append(True)
    else:
        tofs.append(None)
        possible.append(False)

valid_hdg = [h for h, p in zip(headings, possible) if p]
valid_tof = [t for t, p in zip(tofs,     possible) if p]
none_hdg  = [h for h, p in zip(headings, possible) if not p]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(valid_hdg, valid_tof, "o-", color="#2a9d8f", markersize=4, linewidth=1.5,
        label="Intercept possible")
if none_hdg:
    ax.scatter(none_hdg, [max(valid_tof) * 1.1] * len(none_hdg),
               marker="x", color="#e63946", s=50, zorder=5, label="Impossible (None)")
ax.set_xlabel("Target heading (°)")
ax.set_ylabel("Time of flight (min)")
ax.set_title("Intercept time vs. target heading  (shooter 600 km/h, target 300 km/h)")
ax.set_xticks(range(0, 361, 30))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

if valid_tof:
    best_idx = valid_tof.index(min(valid_tof))
    print(f"Shortest TOF: {min(valid_tof):.2f} min at heading {valid_hdg[best_idx]}°")
print(f"Impossible headings: {none_hdg if none_hdg else 'none'}")

## Exercise C — Multi-Target Scramble

You have **three targets** departing simultaneously from different positions. The shooter is at Wichita Falls. You can only engage one target at a time — you must decide which to fire on first.

Rank the targets by intercept time and display both pursuit and intercept paths for all three on one map. Use a different color per target.

**Targets:**

| Target | Position | Heading | Speed |
|--------|----------|---------|-------|
| A      | `[-101.0, 36.5]` | 90°  | 250 km/h |
| B      | `[-96.5, 35.8]`  | 200° | 350 km/h |
| C      | `[-99.0, 37.5]`  | 45°  | 200 km/h |

Shooter speed: 700 km/h.

Answer: Which target should be engaged first, and why? Does the pursuer always agree with the interceptor on the ranking?

In [ ]:
from ipyleaflet import Map, GeoJSON

s_pos  = [-98.49, 33.91]
s_spd  = 700

targets = [
    {"label": "A", "pos": [-101.0, 36.5], "hdg": 90,  "spd": 250, "color": "#e63946"},
    {"label": "B", "pos": [-96.5,  35.8], "hdg": 200, "spd": 350, "color": "#f4a261"},
    {"label": "C", "pos": [-99.0,  37.5], "hdg": 45,  "spd": 200, "color": "#2a9d8f"},
]

multi_map = Map(center=(35.5, -99.0), zoom=6)

def add_fc_layer(m, features, style=None, point_style=None):
    kwargs = {"data": {"type": "FeatureCollection", "features": features}}
    if style:       kwargs["style"]       = style
    if point_style: kwargs["point_style"] = point_style
    m.add(GeoJSON(**kwargs))

results = []
for tgt in targets:
    t_pos = tgt["pos"]
    t_hdg = tgt["hdg"]
    t_spd = tgt["spd"]
    color = tgt["color"]

    # Intercept
    t_int = find_intercept_time(s_pos, t_pos, t_hdg, t_spd, s_spd)
    if t_int is not None:
        ipt = destination_point(t_pos, t_hdg, t_spd * t_int)
        add_fc_layer(multi_map,
            [{"type":"Feature","geometry":{"type":"LineString","coordinates":[s_pos, ipt]},"properties":{}}],
            style={"color": color, "weight": 2, "dashArray": "6"})
        add_fc_layer(multi_map,
            [{"type":"Feature","geometry":{"type":"Point","coordinates": ipt},"properties":{}}],
            point_style={"radius": 6, "color": color, "fillOpacity": 1.0})

    # Pursuit
    sim = simulate_pursuit(s_pos, t_pos, t_hdg, t_spd, s_spd, dt=0.01, max_steps=3000)
    step = max(1, len(sim["pursuer_path"]) // 200)
    add_fc_layer(multi_map,
        [{"type":"Feature","geometry":{"type":"LineString",
          "coordinates": sim["target_path"][::step]},"properties":{}}],
        style={"color": color, "weight": 1.0, "dashArray": "3", "opacity": 0.5})
    add_fc_layer(multi_map,
        [{"type":"Feature","geometry":{"type":"LineString",
          "coordinates": sim["pursuer_path"][::step]},"properties":{}}],
        style={"color": color, "weight": 2.0})

    results.append({
        "label":    tgt["label"],
        "intercept_tof": t_int * 60 if t_int else None,
        "pursuit_tof":   sim["time_elapsed"] * 60 if sim["captured"] else None,
    })

# Shooter marker
add_fc_layer(multi_map,
    [{"type":"Feature","geometry":{"type":"Point","coordinates": s_pos},"properties":{}}],
    point_style={"radius": 9, "color": "#000", "fillOpacity": 1.0})

# Rankings
print("Target  Intercept TOF    Pursuit TOF")
print("-" * 42)
for r in sorted(results, key=lambda x: x["intercept_tof"] or 9999):
    i_str = f"{r['intercept_tof']:.2f} min" if r["intercept_tof"] else "impossible"
    p_str = f"{r['pursuit_tof']:.2f} min"   if r["pursuit_tof"]   else "no capture"
    print(f"  {r['label']}     {i_str:<16}  {p_str}")

multi_map

---

## Check Your Understanding

A student runs the simulation with a shooter speed of 500 km/h and a target speed of 400 km/h. The `find_intercept_time` function returns `None`. The student concludes:

> *"The shooter is faster, so the intercept must be possible. I must have a bug in my code."*

**Question:** Is the student's reasoning correct? When can a faster shooter still fail to achieve an intercept? What would you tell this student to check?

```python
# your answer here
```

## Next

In [04 — Strategy and Limits](./04-Strategy_and_Limits.ipynb), we examine the boundary conditions of both approaches — escape cones, minimum speed ratios, and the geometry that determines whether any intercept solution exists.